# App Event Data Conversion & EDA

This notebook converts raw app event CSV files (located in `data/raw/202503_app_event/`) into Parquet format and performs basic Exploratory Data Analysis (EDA).

## 1. Environment Setup

In [1]:
import polars as pl
import os
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
RAW_DIR = Path("../data/raw/202503_app_event/")
PROCESSED_DIR = Path("../data/processed/app_event/")

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

Raw data directory: ..\data\raw\202503_app_event
Processed data directory: ..\data\processed\app_event


## 2. CSV to Parquet Conversion

We use `polars` for efficient handling of large CSV files. We will read them as lazy dataframes and then sink them into Parquet files.

In [2]:
csv_files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"Found {len(csv_files)} CSV files to process.")

for csv_path in csv_files:
    file_name = Path(csv_path).stem
    output_path = PROCESSED_DIR / f"{file_name}.parquet"
    
    if output_path.exists():
        print(f"Skipping {file_name}.csv, already exists at {output_path}")
        continue
        
    print(f"Processing: {file_name}.csv ...")
    
    try:
        # Use scan_csv for memory efficiency with large files
        df = pl.scan_csv(
            csv_path, 
            ignore_errors=True, 
            infer_schema_length=10000, 
            truncate_ragged_lines=True
        )
        
        # Materialize and save to parquet
        df.collect().write_parquet(output_path)
        print(f"Successfully saved to: {output_path}")
        
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

Found 0 CSV files to process.


## 3. Basic EDA

Now let's look at the summary of the processed data.

In [5]:
parquet_files = glob.glob(str(PROCESSED_DIR / "*.parquet"))

for pq_path in parquet_files:
    file_name = Path(pq_path).stem
    df = pl.read_parquet(pq_path)
    
    print(f"\n--- Analysis of {file_name} ---")
    print(f"Shape: {df.shape}")
    
    # 1. Event Name Distribution
    if "Event Name" in df.columns:
        event_counts = df["Event Name"].value_counts().sort("count", descending=True).head(15)
        
        plt.figure(figsize=(12, 6))
        sns.barplot(x="count", y="Event Name", data=event_counts.to_pandas())
        plt.title(f"Top 15 Event Names - {file_name}")
        plt.show()
    
    # 2. Time Range (if Event Time exists)
    if "Event Time" in df.columns:
        try:
            # Try to cast to datetime if not already
            event_times = df["Event Time"].cast(pl.Datetime, strict=False)
            min_time = event_times.min()
            max_time = event_times.max()
            print(f"Event Time Range: {min_time} ~ {max_time}")
        except:
            print("Could not parse Event Time as datetime.")
            
    # 3. Media Source Analysis
    if "Media Source" in df.columns:
        media_counts = df["Media Source"].value_counts().sort("count", descending=True).head(5)
        print("Top 5 Media Sources:")
        print(media_counts)
        
    # 4. Missing values check
    null_counts = df.null_count().to_pandas().T
    null_counts.columns = ['null_count']
    print("Columns with missing values:")
    print(null_counts[null_counts['null_count'] > 0])


In [7]:
PROCESSED_DIR = Path("../data/processed/app_event/")
parquet_files = glob.glob(str(PROCESSED_DIR / "*.parquet"))

if not parquet_files:
    print(f"경고: {PROCESSED_DIR.absolute()} 경로에 Parquet 파일이 없습니다.")
    print("먼저 CSV를 Parquet으로 변환하는 과정을 완료했는지 확인해 주세요.")
else:
    print(f"--- 앱 이벤트 데이터 기간 분석 시작 ---")
    print(f"분석 대상 파일 수: {len(parquet_files)}개\n")

    all_stats = []

    for pq_path in parquet_files:
        file_name = Path(pq_path).name
        
        try:
            # 레이지 로딩으로 데이터 스캔 (메모리 효율적)
            lf = pl.scan_parquet(pq_path)
            
            # Event Time 컬럼 존재 여부 확인
            columns = lf.collect_schema().names()
            if "Event Time" in columns:
                # 시간 데이터 처리 및 최소/최대값 계산
                # 문자열인 경우 Datetime으로 변환 시도
                date_series = df["Event Time"].str.slice(0, 10)
                valid_dates = date_series.str.to_date(strict=False)
                start = valid_dates.min()
                end = valid_dates.max()
                count = len(df)
                
                if start and end:
                    all_stats.append({"file": file_name, "start": start, "end": end, "count": count})
                    print(f"[{file_name}]")
                    print(f"  - 기간: {start} ~ {end}")
                    print(f"  - 데이터 수: {count:,} 건")
                    
                    # 3월 데이터 여부 확인
                    if start.month == 3 or end.month == 3:
                        print(f"  => [확인] 2025년 3월 데이터가 포함되어 있습니다.")
                else:
                    # 파싱 실패 시 샘플 데이터 출력하여 원인 파악
                    sample = df["Event Time"].head(3).to_list()
                    print(f"[{file_name}] 날짜 추출 실패. 샘플 데이터: {sample}")
            else:
                print(f"[{file_name}] 'Event Time' 컬럼이 없습니다.")

        except Exception as e:
            print(f"[{file_name}] 처리 중 오류: {e}")

    # 2. 전체 통합 기간 계산
    if all_stats:
        overall_start = min(s["start"] for s in all_stats)
        overall_end = max(s["end"] for s in all_stats)
        total_count = sum(s["count"] for s in all_stats)
        
        print(f"\n" + "="*60)
        print(f"전체 통합 분석 결과")
        print(f"="*60)
        print(f"최초 이벤트 발생일: {overall_start}")
        print(f"최종 이벤트 발생일: {overall_end}")
        
        duration = overall_end - overall_start
        print(f"총 데이터 기간: {duration.days}일 {duration.seconds // 3600}시간")
        print(f"총 이벤트 건수: {total_count:,} 건")
        print(f"="*60)
    else:
        print("\n분석 가능한 데이터가 없습니다.")


경고: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\eda\notebooks\..\data\processed\app_event 경로에 Parquet 파일이 없습니다.
먼저 CSV를 Parquet으로 변환하는 과정을 완료했는지 확인해 주세요.


## 4. Combined Summary

Check total volume of app events.

In [ ]:
total_rows = 0
for pq_path in parquet_files:
    total_rows += pl.scan_parquet(pq_path).select(pl.len()).collect().item()

print(f"Total App Events across all files: {total_rows:,}")